In [0]:
from pyspark.sql.functions import col , lower, initcap, month, year, when, count, regexp_replace
from pyspark.sql.types import DoubleType, IntegerType
     

In [0]:
# acessando a tabela delta Bronze

df_bronze = spark.table(
    "projeto_profissional_bigdata.datatran_2018.bronze_datatran18"
)

In [0]:

df_bronze.printSchema()

In [0]:
# selecionando colunas

df_silver = df_bronze.select(
    "id",
    "data_inversa",
    "uf",
    "br",
    "km",
    "municipio",
    "causa_acidente",
    "tipo_acidente"
)

df_silver.show(5)
df_silver.printSchema()
     

In [0]:
# removendo linhas com valores nulos

df_silver = df_silver.dropna(
  subset=["id", "data_inversa", "uf", "br", "km", "municipio", "causa_acidente", "tipo_acidente"]  
)

df_silver.show(5, truncate= False)
     


In [0]:
# renomeando coluna data_inversa para data_acidente

df_silver = df_silver.withColumnRenamed("data_inversa", "data_acidente")

In [0]:

# Padronizando nome de colunas

df_silver = df_silver.withColumn(
    "municipio",
    initcap(col("municipio"))
)
df_silver = df_silver.withColumn(
    "causa_acidente",
    lower(col("causa_acidente"))
)
df_silver = df_silver.withColumn(
    "tipo_acidente",
    lower(col("tipo_acidente"))
)

df_silver.show(5)

In [0]:
# criando coluna mes_acidente

df_silver = df_silver.withColumn(
    "mes_acidente",
    month(col("data_acidente"))
)

In [0]:
# criando a coluna ano acidente

df_silver = df_silver.withColumn(
    "ano_acidente",
    year(col("data_acidente"))
)

df_silver.show(5)


In [0]:
df_silver = df_silver.withColumn(
    "nome_mes",
    when(col("mes_acidente") == 1, "Janeiro")
    .when(col("mes_acidente") == 2, "Fevereiro")
    .when(col("mes_acidente") == 3, "Março")
    .when(col("mes_acidente") == 4, "Abril")
    .when(col("mes_acidente") == 5, "Maio")
    .when(col("mes_acidente") == 6, "Junho")
    .when(col("mes_acidente") == 7, "Julho")
    .when(col("mes_acidente") == 8, "Agosto")
    .when(col("mes_acidente") == 9, "Setembro")
    .when(col("mes_acidente") == 10, "Outubro")
    .when(col("mes_acidente") == 11, "Novembro")
    .otherwise("Dezembro")
)

df_silver.show(5)

In [0]:
# Contando linhas da coluna br = NA

df_silver.filter(col("br") == "NA").count()

In [0]:
# Trocando o valor da coluna br = NA por None

df_silver = df_silver.withColumn(
    "br",
    when(col("br") == "NA", None)
    .otherwise(col("br"))
)

In [0]:
# Limpando os dados nulos da coluna br

df_silver = df_silver.dropna(subset=["br"])

In [0]:
df_silver = df_silver.withColumn(
    "br",
    col("br").cast(IntegerType())
)

df_silver.printSchema()
df_silver.show(5)

In [0]:
# transformando a coluna km em double e trocando os caracteres "," por "."

df_silver = df_silver.withColumn(
    "km",
    regexp_replace(col("km"), ",", ".").cast(DoubleType())
)

df_silver.printSchema()
df_silver.show(5)

In [0]:
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("projeto_profissional_bigdata.datatran_2018.silver_datatran18")

In [0]:
df = spark.table("projeto_profissional_bigdata.datatran_2018.silver_datatran18")

df.show()

In [0]:
%sql
SELECT * FROM projeto_profissional_bigdata.datatran_2018.silver_datatran18